# Game manager: the turn-by-turn state machine

**Deliverable:** the game loop with the camera as the source of truth (README: *"Infer the human's
move from the changed squares and the legal moves, and ask the human when unsure"*), and rule
legality checked separately from physical placement (Perception and rules milestone).

```
START -> (human is white) WAIT_HUMAN -> DETECT_MOVE -> VALIDATE -> ROBOT_THINK -> ROBOT_EXECUTE -> VERIFY -> WAIT_HUMAN ...
                                 \-> RECOVER <-/                                        \-> RECOVER
any state -> GAME_OVER
```

The manager owns the *rules board* (`self.board`, positions confirmed by the camera). Hardware and
policy are injected (dependency injection), so the same class runs on the real robot, in Gazebo
and fully in software (`game_loop.ipynb`):

* `vision`: `VisionSource.read() -> BoardObservation`
* `arm`: `ArmInterface` (physical_moves.ipynb)
* `policy`: `MovePolicy.choose_move(board) -> chess.Move` (the teacher policy from
  `gambit_teach/notebooks/move_selection.ipynb`, or `EnginePolicy`)
* `listeners`: callbacks `fn(event: MoveEvent)` for logging, hesitation features, the player model.

**TODO checklist**
- [x] State enum, transition table, legal-move matching from a camera observation.
- [ ] `on_human_timeout`: what to do when the human does not move (prompt? pause clock?).
- [ ] `resolve_ambiguous`: several legal moves fit the observation (e.g. promotion piece unclear).
- [ ] RECOVER branch: depends on `choose_recovery_action` / `plan_correction` (recovery.ipynb).
- [ ] Human-move "done" signal: stable board only, or also a button / clock press?
- [ ] Draw offers, resignation, takebacks (out of scope for Mark-I?).

In [ ]:
%run ./engine.ipynb
%run ./physical_moves.ipynb
%run ./recovery.ipynb

In [ ]:
import enum
import time
from dataclasses import dataclass, field
from typing import Callable, Optional, Protocol

import chess

GAME_SETTINGS = GAME_CFG["game"]

## States and allowed transitions

The table is the specification: `GameManager.transition` refuses anything not listed, so a bug
shows up as an exception instead of a robot doing something odd.

In [ ]:
class State(enum.Enum):
    START = "start"                  # check the initial board
    WAIT_HUMAN = "wait_human"        # wait for a stable, changed board
    DETECT_MOVE = "detect_move"      # observed board -> legal move
    VALIDATE = "validate"            # rule legality, push the human's move
    ROBOT_THINK = "robot_think"      # policy picks a move, plan arm steps
    ROBOT_EXECUTE = "robot_execute"  # run the arm steps
    VERIFY = "verify"                # camera confirms the robot's move
    RECOVER = "recover"              # mismatch handling
    GAME_OVER = "game_over"


TRANSITIONS: dict[State, set[State]] = {
    State.START: {State.WAIT_HUMAN, State.ROBOT_THINK, State.RECOVER},
    State.WAIT_HUMAN: {State.DETECT_MOVE},
    State.DETECT_MOVE: {State.VALIDATE, State.RECOVER},
    State.VALIDATE: {State.ROBOT_THINK, State.RECOVER},
    State.ROBOT_THINK: {State.ROBOT_EXECUTE},
    State.ROBOT_EXECUTE: {State.VERIFY},
    State.VERIFY: {State.WAIT_HUMAN, State.RECOVER},
    State.RECOVER: {State.START, State.WAIT_HUMAN, State.VERIFY, State.ROBOT_EXECUTE, State.DETECT_MOVE},
    State.GAME_OVER: set(),
}
for _s in State:                         # the game can end from anywhere
    if _s is not State.GAME_OVER:
        TRANSITIONS[_s].add(State.GAME_OVER)

## Injected interfaces

In [ ]:
class VisionSource(Protocol):
    def read(self) -> BoardObservation: ...


class MovePolicy(Protocol):
    def choose_move(self, board: chess.Board) -> chess.Move: ...


class EnginePolicy:
    """Plain strength-limited engine play (the teacher's fallback)."""

    def __init__(self, engine: EngineLike):
        self.engine = engine

    def choose_move(self, board: chess.Board) -> chess.Move:
        return self.engine.play(board)


@dataclass
class MoveEvent:
    ply: int
    by: str                          # "human" or "robot"
    fen_before: str
    move: chess.Move
    san: str
    think_time_s: float
    robot_attempts: int = 0
    plan: str = ""
    observation: Optional[BoardObservation] = None
    extra: dict = field(default_factory=dict)

## Reading the board

`wait_for_stable_board`: the board is trusted only when the same placement is seen in
`stable_frames` consecutive frames with no hand over the board.

`match_legal_moves`: try every legal move and keep those whose resulting placement agrees with the
observation. Squares below the confidence threshold are ignored, so one uncertain square does not
hide an otherwise clear move (but an ignored square never *creates* a match on its own: at least
the move's own from/to squares must be confident).

In [ ]:
class HumanTimeout(Exception):
    pass


def wait_for_stable_board(vision: VisionSource, n_frames: int = GAME_SETTINGS["stable_frames"],
                          timeout_s: float = GAME_SETTINGS["human_move_timeout_s"],
                          clock: Callable[[], float] = time.monotonic,
                          max_reads: int = 100_000) -> BoardObservation:
    start, last, count = clock(), None, 0
    for _ in range(max_reads):
        obs = vision.read()
        if obs.hand_present:
            last, count = None, 0
        elif last is not None and obs.placement == last.placement:
            count += 1
        else:
            last, count = obs, 1
        if count >= n_frames:
            return obs
        if clock() - start > timeout_s:
            raise HumanTimeout()
    raise HumanTimeout()


def match_legal_moves(board: chess.Board, obs: BoardObservation,
                      min_conf: float = MIN_SQUARE_CONFIDENCE) -> list[chess.Move]:
    observed = obs.piece_map()
    uncertain = obs.low_confidence_squares(min_conf)
    matches = []
    for move in board.legal_moves:
        if {move.from_square, move.to_square} & uncertain:
            continue
        b = board.copy(stack=False)
        b.push(move)
        expected = b.piece_map()
        if all(expected.get(sq) == observed.get(sq) for sq in chess.SQUARES if sq not in uncertain):
            matches.append(move)
    return matches

## The manager

In [ ]:
class GameManager:
    def __init__(self, vision: VisionSource, arm: ArmInterface, policy: MovePolicy,
                 human_color: chess.Color = chess.WHITE, board: Optional[chess.Board] = None,
                 listeners: Optional[list] = None, clock: Callable[[], float] = time.monotonic,
                 settings: dict = GAME_SETTINGS, recovery_cfg: dict = RECOVERY_CFG):
        self.vision, self.arm, self.policy = vision, arm, policy
        self.human_color = human_color
        self.board = board or chess.Board()
        self.graveyard = Graveyard()
        self.listeners = listeners or []
        self.clock = clock
        self.settings, self.recovery_cfg = settings, recovery_cfg
        self.state = State.START
        self.trace: list[State] = [State.START]
        # per-move scratch
        self.observation: Optional[BoardObservation] = None
        self.pending_move: Optional[chess.Move] = None
        self.plan: list[Step] = []
        self.expected_after: Optional[chess.Board] = None
        self.recovery_attempts = 0
        self.robot_attempts = 0
        self.turn_started = clock()
        self.robot_think_s = 0.0
        self.mismatch_human_turn = False

    # -- plumbing ----------------------------------------------------------------
    def transition(self, new: State) -> None:
        if new not in TRANSITIONS[self.state]:
            raise RuntimeError(f"illegal transition {self.state.name} -> {new.name}")
        self.state = new
        self.trace.append(new)

    def emit(self, event: MoveEvent) -> None:
        for fn in self.listeners:
            fn(event)

    def human_to_move(self) -> bool:
        return self.board.turn == self.human_color

    def _after_confirmed_move(self) -> None:
        self.recovery_attempts = 0
        self.turn_started = self.clock()
        if self.board.is_game_over(claim_draw=True):
            self.transition(State.GAME_OVER)
        else:
            self.transition(State.WAIT_HUMAN if self.human_to_move() else State.ROBOT_THINK)

    # -- handlers ----------------------------------------------------------------
    def on_start(self):
        obs = wait_for_stable_board(self.vision, self.settings["stable_frames"], clock=self.clock)
        diff = diff_boards(self.board, obs)
        if diff:
            self.observation, self.mismatch_human_turn = obs, False
            self.expected_after = self.board.copy()
            return self.transition(State.RECOVER)
        self.turn_started = self.clock()
        self.transition(State.WAIT_HUMAN if self.human_to_move() else State.ROBOT_THINK)

    def on_wait_human(self):
        while True:
            try:
                obs = wait_for_stable_board(self.vision, self.settings["stable_frames"],
                                            self.settings["human_move_timeout_s"], self.clock)
            except HumanTimeout:
                self.on_human_timeout()
                continue
            if diff_boards(self.board, obs):
                self.observation = obs
                return self.transition(State.DETECT_MOVE)

    def on_human_timeout(self):
        # TODO: prompt the human (sound/LED), maybe log a hesitation event; decide whether to keep
        # waiting forever or pause the game.
        raise NotImplementedError("on_human_timeout")

    def on_detect_move(self):
        moves = match_legal_moves(self.board, self.observation, self.settings["min_square_confidence"])
        if len(moves) == 1:
            self.pending_move = moves[0]
            return self.transition(State.VALIDATE)
        if len(moves) > 1:
            self.pending_move = self.resolve_ambiguous(moves)
            return self.transition(State.VALIDATE)
        self.mismatch_human_turn = True
        self.expected_after = self.board.copy()
        self.transition(State.RECOVER)

    def resolve_ambiguous(self, moves: list[chess.Move]) -> chess.Move:
        # TODO: re-read the board a few times; if still ambiguous ask the human which move it was.
        raise NotImplementedError(f"resolve_ambiguous: {[m.uci() for m in moves]}")

    def on_validate(self):
        move = self.pending_move
        if move not in self.board.legal_moves:          # rule legality, separate from placement
            self.mismatch_human_turn = True
            self.expected_after = self.board.copy()
            return self.transition(State.RECOVER)
        think = self.observation.timestamp - self.turn_started if self.observation else 0.0
        event = MoveEvent(self.board.ply(), "human", self.board.fen(), move, self.board.san(move),
                          max(0.0, think), observation=self.observation)
        self.graveyard = self._graveyard_after(move)
        self.board.push(move)
        self.emit(event)
        self.recovery_attempts = 0
        self.turn_started = self.clock()
        if self.board.is_game_over(claim_draw=True):
            return self.transition(State.GAME_OVER)
        self.transition(State.ROBOT_THINK)

    def _graveyard_after(self, move: chess.Move) -> Graveyard:
        """Track what the human's move put into the graveyard (the human hands captures to it)."""
        gy = self.graveyard.copy()
        try:
            plan_physical_move(self.board, move, gy)
        except MissingPieceError:
            pass        # the human promoted with a piece we did not have: fine, the human brought it
        return gy

    def on_robot_think(self):
        t0 = self.clock()
        move = self.policy.choose_move(self.board)
        self.robot_think_s = self.clock() - t0
        self.pending_move = move
        self.plan = plan_physical_move(self.board, move, self.graveyard.copy())
        self.expected_after = self.board.copy()
        self.expected_after.push(move)
        self.robot_attempts = 0
        self.transition(State.ROBOT_EXECUTE)

    def on_robot_execute(self):
        self.robot_attempts += 1
        self.last_execution = execute_plan(self.arm, self.plan)
        self.transition(State.VERIFY)      # verify even after a failure: the camera decides

    def on_verify(self):
        obs = wait_for_stable_board(self.vision, self.settings["stable_frames"], clock=self.clock)
        self.observation = obs
        if diff_boards(self.expected_after, obs):
            self.mismatch_human_turn = False
            return self.transition(State.RECOVER)
        move = self.pending_move
        event = MoveEvent(self.board.ply(), "robot", self.board.fen(), move, self.board.san(move),
                          self.robot_think_s, self.robot_attempts, describe_plan(self.plan), obs)
        plan_physical_move(self.board, move, self.graveyard)     # commit graveyard changes
        self.board.push(move)
        self.emit(event)
        self._after_confirmed_move()

    def on_recover(self):
        diff = diff_boards(self.expected_after, self.observation)
        kind = classify_mismatch(diff, self.observation, human_turn=self.mismatch_human_turn)
        action = choose_recovery_action(kind, self.recovery_attempts, self.recovery_cfg)   # TODO in recovery
        self.recovery_attempts += 1
        if action is RecoveryAction.ACCEPT:
            return self.transition(State.DETECT_MOVE if self.mismatch_human_turn else State.VERIFY)
        if action is RecoveryAction.REVERIFY:
            return self.transition(State.WAIT_HUMAN if self.mismatch_human_turn else State.VERIFY)
        if action is RecoveryAction.ARM_RETRY:
            self.plan = plan_correction(diff, kind)                                         # TODO in recovery
            return self.transition(State.ROBOT_EXECUTE)
        if action is RecoveryAction.ASK_HUMAN:
            self.ask_human(diff, kind)
            return self.transition(State.WAIT_HUMAN if self.mismatch_human_turn else State.VERIFY)
        self.transition(State.GAME_OVER)

    def ask_human(self, diff: BoardDiff, kind: MismatchKind) -> None:
        # TODO: tell the human what is wrong (which squares) and wait for a confirmation signal.
        raise NotImplementedError(f"ask_human: {kind.value}: {diff}")

    HANDLERS = {
        State.START: on_start, State.WAIT_HUMAN: on_wait_human, State.DETECT_MOVE: on_detect_move,
        State.VALIDATE: on_validate, State.ROBOT_THINK: on_robot_think,
        State.ROBOT_EXECUTE: on_robot_execute, State.VERIFY: on_verify, State.RECOVER: on_recover,
    }

    def step(self) -> State:
        self.HANDLERS[self.state](self)
        return self.state

    def run(self, max_plies: Optional[int] = None, max_steps: int = 10_000) -> chess.Board:
        """Run until GAME_OVER, or until `max_plies` half-moves have been confirmed."""
        for _ in range(max_steps):
            if self.state is State.GAME_OVER:
                break
            if max_plies is not None and self.board.ply() >= max_plies and self.state in (
                    State.WAIT_HUMAN, State.ROBOT_THINK):
                break
            self.step()
        return self.board

## Self-check (no hardware: a scripted vision source)

In [ ]:
class _ScriptedVision:
    def __init__(self, observations):
        self.obs, self.i = list(observations), 0

    def read(self):
        o = self.obs[min(self.i, len(self.obs) - 1)]
        self.i += 1
        return o

# every listed transition target is a real state, GAME_OVER is terminal
assert all(isinstance(t, State) for ts in TRANSITIONS.values() for t in ts)
assert TRANSITIONS[State.GAME_OVER] == set()

# matching a human move from an observation
b = chess.Board(); after = b.copy(); after.push_san("Nf3")
assert match_legal_moves(b, BoardObservation.from_board(after)) == [chess.Move.from_uci("g1f3")]
conf = [1.0] * 64; conf[chess.H7] = 0.2               # an unrelated uncertain square is ignored
assert match_legal_moves(b, BoardObservation(after.board_fen(), conf)) == [chess.Move.from_uci("g1f3")]
conf = [1.0] * 64; conf[chess.F3] = 0.2               # but the move's own square must be confident
assert match_legal_moves(b, BoardObservation(after.board_fen(), conf)) == []
bad = chess.BaseBoard(after.board_fen()); bad.set_piece_at(chess.F4, chess.Piece.from_symbol("N"))
assert match_legal_moves(b, BoardObservation.from_board(bad)) == []

# stable-board detection ignores frames with a hand and needs n identical frames
o1, o2 = BoardObservation.from_board(b), BoardObservation.from_board(after)
hand = BoardObservation(after.board_fen(), hand_present=True)
v = _ScriptedVision([o1, hand, o2, o2, o2])
assert wait_for_stable_board(v, n_frames=3).placement == o2.placement and v.i == 5

# transition guard
gm = GameManager(_ScriptedVision([o1]), arm=None, policy=None)
try:
    gm.transition(State.VERIFY)
    raise AssertionError("guard failed")
except RuntimeError:
    pass
# START -> WAIT_HUMAN -> DETECT_MOVE -> VALIDATE -> ROBOT_THINK with a scripted human move
gm = GameManager(_ScriptedVision([o1] * 5 + [o2] * 5), arm=None, policy=None)
for _ in range(4):
    gm.step()
assert gm.trace == [State.START, State.WAIT_HUMAN, State.DETECT_MOVE, State.VALIDATE, State.ROBOT_THINK], gm.trace
assert gm.board.move_stack == [chess.Move.from_uci("g1f3")]
print("game_manager self-check passed")